# LightGBM Feature-Engineered Baseline for Spaceship Titanic

这个 notebook 使用队友提供的 `spaceship_catboost_preprocessed_package.zip` 作为输入，先训练一个 LightGBM baseline。

目标：
- 读取已经完成强特征工程的训练集和测试集
- 正确处理 LightGBM 的 categorical features
- 使用 5-fold StratifiedKFold 做本地验证
- 搜索最优分类阈值
- 输出 Kaggle submission 文件
- 保存 OOF/test probability，方便后续做 soft voting ensemble

把这个 notebook 和压缩包放在同一个文件夹里运行即可。


In [ ]:
import os
import json
import time
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")

try:
    import lightgbm as lgb
    print("LightGBM version:", lgb.__version__)
except ImportError as e:
    raise ImportError("LightGBM is not installed. Please run: pip install lightgbm") from e


## Load the preprocessed package

默认情况下，notebook 会在当前文件夹寻找：

`spaceship_catboost_preprocessed_package.zip`

如果你在本地运行，只需要把 zip 和 notebook 放在一起。


In [ ]:
ZIP_PATH = Path("spaceship_catboost_preprocessed_package.zip")
EXTRACT_DIR = Path("spaceship_catboost_preprocessed_package")

if not ZIP_PATH.exists():
    backup_path = Path("/mnt/data/spaceship_catboost_preprocessed_package.zip")
    if backup_path.exists():
        ZIP_PATH = backup_path
    else:
        raise FileNotFoundError(
            "Cannot find spaceship_catboost_preprocessed_package.zip. "
            "Please put it in the same folder as this notebook."
        )

if not EXTRACT_DIR.exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)

print("Using zip file:", ZIP_PATH)
print("Extracted files:")
for file_name in sorted(os.listdir(EXTRACT_DIR)):
    print("-", file_name)


In [ ]:
X_train = pd.read_csv(EXTRACT_DIR / "X_train_catboost_features.csv", low_memory=False)
X_test = pd.read_csv(EXTRACT_DIR / "X_test_catboost_features.csv", low_memory=False)
y_df = pd.read_csv(EXTRACT_DIR / "y_train_with_ids.csv")
test_ids = pd.read_csv(EXTRACT_DIR / "test_passenger_ids.csv")

with open(EXTRACT_DIR / "catboost_preprocessing_metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

y = y_df["Transported"].astype(int)

categorical_cols = [col for col in metadata["categorical_columns"] if col in X_train.columns]
missing_categorical_cols = [col for col in metadata["categorical_columns"] if col not in X_train.columns]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y shape:", y.shape)
print("Number of categorical columns used:", len(categorical_cols))

if missing_categorical_cols:
    print("Categorical columns not found:", missing_categorical_cols)
else:
    print("All metadata categorical columns are found.")

display(X_train.head())


## Prepare LightGBM categorical features

LightGBM 不能直接把普通字符串列当成数值特征使用。这里把 metadata 里记录的类别列统一转成 `category` 类型，并保证 train/test 使用同一套 categories。


In [ ]:
def align_categorical_columns(train_df, test_df, cat_cols):
    train_df = train_df.copy()
    test_df = test_df.copy()

    for col in cat_cols:
        train_df[col] = train_df[col].astype("string").fillna("Missing")
        test_df[col] = test_df[col].astype("string").fillna("Missing")

        all_categories = pd.Index(pd.concat([train_df[col], test_df[col]], axis=0).unique())
        train_df[col] = pd.Categorical(train_df[col], categories=all_categories)
        test_df[col] = pd.Categorical(test_df[col], categories=all_categories)

    return train_df, test_df


X_train_lgb, X_test_lgb = align_categorical_columns(X_train, X_test, categorical_cols)

print("Object columns after conversion:")
print(X_train_lgb.select_dtypes(include=["object"]).columns.tolist())

print("\nCategorical dtypes:")
print(X_train_lgb[categorical_cols].dtypes.head(10))


## 5-fold LightGBM baseline

这里先不做 Optuna，先用一组稳健参数跑出 baseline。拿到 baseline 后，再决定是调参、删特征，还是进入 ensemble。


In [ ]:
N_SPLITS = 5
RANDOM_STATE = 42

lgb_params = {
    "objective": "binary",
    "boosting_type": "gbdt",
    "n_estimators": 5000,
    "learning_rate": 0.015,
    "num_leaves": 31,
    "max_depth": 5,
    "min_child_samples": 20,
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": -1,
}

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

oof_pred = np.zeros(len(X_train_lgb))
test_pred = np.zeros(len(X_test_lgb))
fold_results = []
models = []
feature_gain_list = []

start_total = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_lgb, y), start=1):
    X_tr = X_train_lgb.iloc[train_idx]
    X_val = X_train_lgb.iloc[val_idx]
    y_tr = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    model = lgb.LGBMClassifier(**lgb_params)

    start_fold = time.time()
    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="binary_logloss",
        categorical_feature=categorical_cols,
        callbacks=[
            lgb.early_stopping(stopping_rounds=150, verbose=False),
            lgb.log_evaluation(period=250),
        ],
    )
    fold_time = time.time() - start_fold

    val_proba = model.predict_proba(X_val)[:, 1]
    test_proba = model.predict_proba(X_test_lgb)[:, 1]

    oof_pred[val_idx] = val_proba
    test_pred += test_proba / N_SPLITS

    val_label_05 = (val_proba >= 0.5).astype(int)
    fold_acc = accuracy_score(y_val, val_label_05)

    best_iteration = model.best_iteration_ if model.best_iteration_ else lgb_params["n_estimators"]

    fold_results.append({
        "fold": fold,
        "accuracy_threshold_0.5": fold_acc,
        "best_iteration": best_iteration,
        "training_time_seconds": fold_time,
    })

    gain = model.booster_.feature_importance(importance_type="gain")
    feature_gain_list.append(gain)

    models.append(model)

    print(
        f"Fold {fold}: accuracy@0.5 = {fold_acc:.5f}, "
        f"best_iteration = {best_iteration}, time = {fold_time:.2f}s"
    )

total_time = time.time() - start_total

fold_results_df = pd.DataFrame(fold_results)
display(fold_results_df)

cv_acc_05 = accuracy_score(y, (oof_pred >= 0.5).astype(int))

print(f"\nOOF accuracy at threshold 0.5: {cv_acc_05:.5f}")
print(f"Total CV training time: {total_time:.2f}s")


## Search the best classification threshold

Kaggle 评价的是 accuracy，不一定固定用 0.5 阈值最好。这里用 OOF probability 搜索最优阈值。


In [ ]:
thresholds = np.arange(0.35, 0.651, 0.001)

threshold_scores = []
for threshold in thresholds:
    pred_label = (oof_pred >= threshold).astype(int)
    acc = accuracy_score(y, pred_label)
    threshold_scores.append(acc)

threshold_scores = np.array(threshold_scores)
best_idx = threshold_scores.argmax()
best_threshold = thresholds[best_idx]
best_oof_acc = threshold_scores[best_idx]

print(f"Best threshold: {best_threshold:.3f}")
print(f"Best OOF accuracy: {best_oof_acc:.5f}")
print(f"OOF accuracy at 0.5: {cv_acc_05:.5f}")

plt.figure(figsize=(8, 4))
plt.plot(thresholds, threshold_scores)
plt.axvline(best_threshold, linestyle="--")
plt.xlabel("Threshold")
plt.ylabel("OOF Accuracy")
plt.title("Threshold Search for LightGBM")
plt.show()


In [ ]:
oof_label = (oof_pred >= best_threshold).astype(int)

print("Classification report based on best threshold:")
print(classification_report(y, oof_label, target_names=["Not Transported", "Transported"]))

print("Confusion matrix:")
print(confusion_matrix(y, oof_label))


## Feature importance

这里用 LightGBM 的 gain importance，看模型主要依赖哪些特征。


In [ ]:
mean_gain = np.mean(np.vstack(feature_gain_list), axis=0)

importance_df = pd.DataFrame({
    "feature": X_train_lgb.columns,
    "importance_gain": mean_gain,
}).sort_values("importance_gain", ascending=False)

display(importance_df.head(30))

plt.figure(figsize=(8, 7))
top_n = 25
plt.barh(
    importance_df.head(top_n)["feature"][::-1],
    importance_df.head(top_n)["importance_gain"][::-1],
)
plt.xlabel("Mean Gain Importance")
plt.title("Top LightGBM Feature Importances")
plt.tight_layout()
plt.show()


## Create Kaggle submission

这个文件可以直接提交到 Kaggle。第一次建议先提交 `submission_lgbm_v1.csv`，记录 public score。


In [ ]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

test_label = (test_pred >= best_threshold).astype(bool)

submission = pd.DataFrame({
    "PassengerId": test_ids["PassengerId"],
    "Transported": test_label,
})

submission_path = OUTPUT_DIR / "submission_lgbm_v1.csv"
submission.to_csv(submission_path, index=False)

print("Saved submission to:", submission_path)
display(submission.head())
print(submission["Transported"].value_counts(normalize=True))


## Save probabilities for later ensemble

后面做 LightGBM + CatBoost + XGBoost soft voting 时，这两个 probability 文件会很有用。


In [ ]:
oof_output = pd.DataFrame({
    "PassengerId": y_df["PassengerId"],
    "y_true": y,
    "lgbm_oof_proba": oof_pred,
    "lgbm_oof_label_best_threshold": oof_label,
})

test_output = pd.DataFrame({
    "PassengerId": test_ids["PassengerId"],
    "lgbm_test_proba": test_pred,
    "lgbm_test_label_best_threshold": test_label,
})

oof_output_path = OUTPUT_DIR / "lgbm_oof_predictions.csv"
test_output_path = OUTPUT_DIR / "lgbm_test_predictions.csv"
importance_path = OUTPUT_DIR / "lgbm_feature_importance.csv"
fold_results_path = OUTPUT_DIR / "lgbm_fold_results.csv"

oof_output.to_csv(oof_output_path, index=False)
test_output.to_csv(test_output_path, index=False)
importance_df.to_csv(importance_path, index=False)
fold_results_df.to_csv(fold_results_path, index=False)

print("Saved:")
print("-", oof_output_path)
print("-", test_output_path)
print("-", importance_path)
print("-", fold_results_path)


## What to report after running

把下面这些结果发给我，我会根据结果决定下一版怎么调：

- `Best OOF accuracy`
- `Best threshold`
- 每一折的 accuracy 和 best_iteration
- Kaggle public score
- feature importance 前 20 名
- 如果报错，把完整报错发来
